### Import Libraries

In [26]:
import os
import glob
import pandas as pd
from scipy import stats
import seaborn as sns
import pickle
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin

import xgboost as xgb
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, validation_curve, learning_curve
from sklearn.metrics import make_scorer, f1_score, roc_auc_score, roc_curve, precision_recall_curve, f1_score, classification_report, confusion_matrix
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression 
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import r2_score

import warnings 
warnings.filterwarnings("ignore")

import argparse

import model_inference

### Set Up Pyspark Session

In [9]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

### Set Up Config


In [10]:
snapshot_date_str = "2024-01-01"
model_name = "XGB_model_2024_06_01.pkl"


In [11]:
config = {}
config["snapshot_date_str"] = snapshot_date_str
config["snapshot_date"] = datetime.strptime(config["snapshot_date_str"], "%Y-%m-%d")
config["model_name"] = model_name
config["model_bank_directory"] = "model_bank/"
config["model_artefact_filepath"] = config["model_bank_directory"] + config["model_name"]

pprint.pprint(config)

{'model_artefact_filepath': 'model_bank/XGB_model_2024_06_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'XGB_model_2024_06_01.pkl',
 'snapshot_date': datetime.datetime(2024, 1, 1, 0, 0),
 'snapshot_date_str': '2024-01-01'}


In [13]:

# Custom outlier handling transformer
class OutlierHandler(BaseEstimator, TransformerMixin):
    def __init__(self, method='iqr', factor=1.5, cap_method='percentile', lower_percentile=1, upper_percentile=99):
        """
        Custom outlier handler
        
        Parameters:
        - method: 'iqr' for IQR method, 'percentile' for percentile capping
        - factor: IQR factor (default 1.5)
        - cap_method: 'percentile' or 'iqr' for capping method
        - lower_percentile/upper_percentile: percentiles for capping (1-99)
        """
        self.method = method
        self.factor = factor
        self.cap_method = cap_method
        self.lower_percentile = lower_percentile
        self.upper_percentile = upper_percentile
        self.bounds_ = {}
    
    def fit(self, X, y=None):
        X_df = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X
        
        for col_idx in range(X_df.shape[1]):
            col_data = X_df.iloc[:, col_idx]
            
            if self.cap_method == 'iqr':
                Q1 = col_data.quantile(0.25)
                Q3 = col_data.quantile(0.75)
                IQR = Q3 - Q1
                lower_bound = Q1 - self.factor * IQR
                upper_bound = Q3 + self.factor * IQR
            else:  # percentile method
                lower_bound = col_data.quantile(self.lower_percentile / 100)
                upper_bound = col_data.quantile(self.upper_percentile / 100)
            
            self.bounds_[col_idx] = (lower_bound, upper_bound)
        
        return self
    
    def transform(self, X):
        X_df = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X.copy()
        
        for col_idx in range(X_df.shape[1]):
            if col_idx in self.bounds_:
                lower_bound, upper_bound = self.bounds_[col_idx]
                X_df.iloc[:, col_idx] = np.clip(X_df.iloc[:, col_idx], lower_bound, upper_bound)
        
        return X_df.values if not isinstance(X, pd.DataFrame) else X_df

# Simple log transformation skewness handler
class LogSkewnessHandler(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.5):
        """
        Simple skewness handler using log transformation
        
        Parameters:
        - threshold: skewness threshold above which to apply log transformation
        """
        self.threshold = threshold
        self.apply_transform_ = {}
        self.shift_values_ = {}
        
    def fit(self, X, y=None):
        from scipy import stats
        
        X_df = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X
        
        for col_idx in range(X_df.shape[1]):
            col_data = X_df.iloc[:, col_idx].dropna()
            skewness = abs(stats.skew(col_data))
            
            if skewness > self.threshold:
                min_val = col_data.min()
                
                if min_val > 0:
                    # Can use log directly
                    self.apply_transform_[col_idx] = True
                    self.shift_values_[col_idx] = 0
                elif min_val >= 0:
                    # Use log1p for non-negative values (handles zeros)
                    self.apply_transform_[col_idx] = True
                    self.shift_values_[col_idx] = 0
                else:
                    # Shift to make all values positive, then log
                    self.apply_transform_[col_idx] = True
                    self.shift_values_[col_idx] = abs(min_val) + 1
            else:
                self.apply_transform_[col_idx] = False
                self.shift_values_[col_idx] = 0
        
        return self
    
    def transform(self, X):
        X_df = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X.copy()
        
        for col_idx in range(X_df.shape[1]):
            if self.apply_transform_.get(col_idx, False):
                try:
                    shift = self.shift_values_[col_idx]
                    
                    if shift > 0:
                        # Shift values to make positive, then log
                        X_df.iloc[:, col_idx] = np.log(X_df.iloc[:, col_idx] + shift)
                    else:
                        # Use log1p for non-negative values, log for positive
                        min_val = X_df.iloc[:, col_idx].min()
                        if min_val >= 0:
                            X_df.iloc[:, col_idx] = np.log1p(X_df.iloc[:, col_idx])
                        else:
                            X_df.iloc[:, col_idx] = np.log(X_df.iloc[:, col_idx])
                            
                except Exception as e:
                    print(f"Warning: Log transform failed for column {col_idx}: {str(e)}")
                    # Keep original values if transformation fails
                    pass
        
        return X_df.values if not isinstance(X, pd.DataFrame) else X_df
    
    def inverse_transform(self, X):
        """
        Inverse transform to get back original scale
        """
        X_df = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X.copy()
        
        for col_idx in range(X_df.shape[1]):
            if self.apply_transform_.get(col_idx, False):
                try:
                    shift = self.shift_values_[col_idx]
                    
                    if shift > 0:
                        # Reverse: exp then subtract shift
                        X_df.iloc[:, col_idx] = np.exp(X_df.iloc[:, col_idx]) - shift
                    else:
                        # Reverse log1p or log
                        if hasattr(self, '_used_log1p') and self._used_log1p.get(col_idx, False):
                            X_df.iloc[:, col_idx] = np.expm1(X_df.iloc[:, col_idx])
                        else:
                            X_df.iloc[:, col_idx] = np.exp(X_df.iloc[:, col_idx])
                            
                except Exception as e:
                    print(f"Warning: Inverse transform failed for column {col_idx}: {str(e)}")
                    pass
        
        return X_df.values if not isinstance(X, pd.DataFrame) else X_df

In [14]:
# Load the model from the pickle file
with open(config["model_artefact_filepath"], 'rb') as file:
    model_artefact = pickle.load(file)

print("Model loaded successfully! " + config["model_artefact_filepath"])

Model loaded successfully! model_bank/XGB_model_2024_06_01.pkl


### Load Feature Store

In [ ]:
feature_location = "datamart/gold/feature_store"

# Load parquet into DataFrame - connect to feature store
features_store_sdf = spark.read.parquet(feature_location)
print("row_count:",features_store_sdf.count())

# extract feature store
features_sdf = features_store_sdf.filter((col("snapshot_date") == config["snapshot_date"]))
print("extracted features_sdf", features_sdf.count(), config["snapshot_date"])

features_pdf = features_sdf.toPandas()
features_pdf

row_count: 8974


extracted features_sdf 485 2024-01-01 00:00:00


,Customer_ID,Name,SSN,snapshot_date,Age,occupation_developer,occupation_scientist,occupation_engineer,occupation_teacher,occupation_manager,...,avg_fe_11,avg_fe_12,avg_fe_13,avg_fe_14,avg_fe_15,avg_fe_16,avg_fe_17,avg_fe_18,avg_fe_19,avg_fe_20
0,CUS_0x1c7e,Aruna Viswanathaf,972-18-0008,2024-01-01,30.0,0,0,0,1,0,...,105.750000,94.250000,101.250000,103.250000,56.750000,67.750000,94.500000,86.666667,93.083333,88.583333
1,CUS_0x247c,Sn,478-48-3295,2024-01-01,48.0,1,0,0,0,0,...,101.833333,112.083333,91.833333,134.083333,122.833333,128.500000,98.500000,109.166667,127.416667,104.000000
2,CUS_0x4d3,enk,336-50-3957,2024-01-01,25.0,0,1,0,0,0,...,101.750000,91.833333,160.583333,92.333333,117.833333,114.666667,60.250000,60.166667,181.500000,82.500000
3,CUS_0x54d7,Paritoshe,052-22-1460,2024-01-01,37.0,0,0,0,0,0,...,84.083333,57.333333,109.833333,146.666667,85.416667,60.250000,126.750000,106.416667,89.250000,55.750000
4,CUS_0x6950,Clarkej,022-35-6033,2024-01-01,55.0,0,0,0,0,0,...,114.500000,97.250000,127.666667,121.583333,61.083333,111.583333,110.000000,61.916667,89.333333,107.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
480,CUS_0x8234,Annh,838-10-8480,2024-01-01,44.0,0,0,0,0,0,...,86.500000,145.833333,75.916667,46.083333,121.416667,94.000000,129.250000,128.166667,168.000000,94.083333
481,CUS_0x7dac,Jenniferx,None,2024-01-01,52.0,0,0,0,1,0,...,84.000000,89.333333,104.000000,93.166667,60.583333,84.000000,68.500000,93.916667,135.333333,125.416667
482,CUS_0x7eb0,Hummelg,None,2024-01-01,49.0,0,0,0,0,0,...,171.833333,99.416667,61.333333,150.750000,97.750000,96.750000,107.000000,94.583333,88.750000,131.833333
483,CUS_0x673c,Elziot,None,2024-01-01,40.0,0,1,0,0,0,...,140.333333,86.000000,53.333333,95.833333,126.250000,79.000000,67.083333,167.916667,139.750000,49.166667


### Preprocess Data For Modelling 

In [16]:
# prepare X_inference

feature_cols = ['Age','occupation_developer', 'occupation_scientist', 'occupation_engineer',
    'occupation_teacher', 'occupation_manager', 'occupation_enterpreneur',
    'occupation_mechanic', 'occupation_musician', 'occupation_architect',
    'occupation_writer', 'occupation_accountant', 'occupation_journalist',
    'occupation_lawyer', 'occupation_doctor', 'occupation_media_manager',
    'occupation_unknown', 'Annual_Income', 'Monthly_Inhand_Salary',
    'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan',
    'Delay_from_due_date', 'Changed_Credit_Limit', 'Outstanding_Debt',
    'Credit_Utilization_Ratio', 'Total_EMI_per_month',
    'Amount_invested_monthly', 'Monthly_Balance', 'Num_of_Delayed_Payment',
    'Credit_Mix_None', 'Credit_Mix_Good', 'Credit_Mix_Standard',
    'Payment_of_Min_Amount_Yes', 'Payment_of_Min_Amount_No',
    'Payment_of_Min_Amount_NM', 'Payment_of_Min_Amount_None',
    'Payment_Behaviour_High_spent_Large_value_payments',
    'Payment_Behaviour_High_spent_Medium_value_payments',
    'Payment_Behaviour_High_spent_Small_value_payments',
    'Payment_Behaviour_Low_spent_Large_value_payments',
    'Payment_Behaviour_Low_spent_Medium_value_payments',
    'Payment_Behaviour_Low_spent_Small_value_payments',
    'Payment_Behaviour_None', 'Loan_Not_Specified',
    'Loan_debt_consolidation_loan', 'Loan_personal_loan',
    'Loan_payday_loan', 'Loan_mortgage_loan', 'Loan_credit_builder_loan',
    'Loan_auto_loan', 'Loan_home_equity_loan', 'Loan_student_loan',
    'disposable_income', 'DTI', 'num_active_loans',
    'credit_history_bucket_Short', 'credit_history_bucket_Medium',
    'credit_history_bucket_Long', 'credit_history_bucket_None',
    'credit_inquiries_bucket_Low', 'credit_inquiries_bucket_Medium',
    'credit_inquiries_bucket_High', 'credit_inquiries_bucket_None',
    'avg_fe_1', 'avg_fe_2', 'avg_fe_3', 'avg_fe_4', 'avg_fe_5', 'avg_fe_6',
    'avg_fe_7', 'avg_fe_8', 'avg_fe_9', 'avg_fe_10', 'avg_fe_11',
    'avg_fe_12', 'avg_fe_13', 'avg_fe_14', 'avg_fe_15', 'avg_fe_16',
    'avg_fe_17', 'avg_fe_18', 'avg_fe_19', 'avg_fe_20'
]

X_inference = features_pdf[feature_cols]
    
# Define column categories based on outlier analysis
numerical_cols = ['Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 
                'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date',   
                'Changed_Credit_Limit', 'Outstanding_Debt', 'Credit_Utilization_Ratio', 
                'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance', 
                'Num_of_Delayed_Payment', 'disposable_income', 'DTI', 'num_active_loans', 
                'avg_fe_1', 'avg_fe_2', 'avg_fe_3', 'avg_fe_4', 'avg_fe_5', 'avg_fe_6', 
                'avg_fe_7', 'avg_fe_8', 'avg_fe_9', 'avg_fe_10', 'avg_fe_11', 'avg_fe_12', 
                'avg_fe_13', 'avg_fe_14', 'avg_fe_15', 'avg_fe_16', 'avg_fe_17', 'avg_fe_18', 
                'avg_fe_19', 'avg_fe_20']

# High VIF features to remove (based on multicollinearity analysis)
high_vif_features = [
    'Monthly_Inhand_Salary', 'credit_history_bucket_Medium', 'num_active_loans', 'credit_inquiries_bucket_Medium'
]

# Drop high VIF features from datasets
X_inference_reduced = X_inference.drop(columns=high_vif_features, errors='ignore')

# Update numerical columns by removing the dropped features
numerical_cols_reduced = [col for col in numerical_cols if col not in high_vif_features]

# Get remaining engineered features (only the ones that weren't dropped)
engineered_features_reduced = [f'avg_fe_{i}' for i in range(1, 21) if f'avg_fe_{i}' not in high_vif_features]

# One-hot encoded columns (categorical features)
all_cols = X_inference.columns.tolist()  # Get all column names from your dataset
onehot_cols = [col for col in all_cols if col not in numerical_cols]

# Update one-hot encoded columns by removing dropped features
onehot_cols_reduced = [col for col in onehot_cols if col not in high_vif_features]

# Categorize features by outlier severity and business logic
# High outlier columns (>5% outliers) - need aggressive treatment
high_outlier_cols = ['Outstanding_Debt', 'Total_EMI_per_month', 'Amount_invested_monthly', 
                    'Monthly_Balance', 'DTI']

# Medium outlier columns (2-5% outliers) - moderate treatment  
# Note: Annual_Income was removed due to high VIF, so excluding it
medium_outlier_cols = ['Monthly_Inhand_Salary', 'disposable_income']

# Low/no outlier columns - minimal treatment
low_outlier_cols = ['Age', 'Interest_Rate', 'Changed_Credit_Limit', 'Credit_Utilization_Ratio']

# Count/discrete variables - business logic validation
count_features = ['Num_of_Loan', 'Num_Credit_Card', 'Num_of_Delayed_Payment', 'Num_Bank_Accounts',
                'Delay_from_due_date', 'num_active_loans']

# Filter all feature categories to only include features that exist in reduced dataset
high_outlier_cols = [col for col in high_outlier_cols if col in numerical_cols_reduced]
medium_outlier_cols = [col for col in medium_outlier_cols if col in numerical_cols_reduced]
low_outlier_cols = [col for col in low_outlier_cols if col in numerical_cols_reduced]
count_features = [col for col in count_features if col in numerical_cols_reduced]

# Fit and transform

# apply preprocessing_transformers from model artefact
preprocessing_transformer = model_artefact["preprocessing_transformers"]["stdscaler"]
X_inference = preprocessing_transformer.transform(X_inference_reduced)

print('X_inference', X_inference.shape[0])
X_inference

X_inference 485


array([[-1.6257062 , -2.43147874, -1.10094714, ...,  1.        ,
         0.        ,  0.        ],
       [-1.80076897,  1.06781495,  1.51130903, ...,  1.        ,
         0.        ,  0.        ],
       [ 0.86382157, -0.08601061, -0.87442613, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [-1.07758594, -0.58504945, -1.14695787, ...,  1.        ,
         0.        ,  0.        ],
       [-1.17256558,  0.57591701,  0.85545206, ...,  1.        ,
         0.        ,  0.        ],
       [ 1.4646033 ,  0.47645459, -2.15631676, ...,  0.        ,
         0.        ,  0.        ]])

### Model Prediction Inference 

In [17]:
# load model
model = model_artefact["model"]

# predict model
y_inference = model.predict_proba(X_inference)[:, 1]

# prepare output
y_inference_pdf = features_pdf[["Customer_ID","snapshot_date"]].copy()
y_inference_pdf["model_name"] = config["model_name"]
y_inference_pdf["model_predictions"] = y_inference
y_inference_pdf

,Customer_ID,snapshot_date,model_name,model_predictions
0,CUS_0x1c7e,2024-01-01,XGB_model_2024_06_01.pkl,0.108502
1,CUS_0x247c,2024-01-01,XGB_model_2024_06_01.pkl,0.106610
2,CUS_0x4d3,2024-01-01,XGB_model_2024_06_01.pkl,0.920999
3,CUS_0x54d7,2024-01-01,XGB_model_2024_06_01.pkl,0.029328
4,CUS_0x6950,2024-01-01,XGB_model_2024_06_01.pkl,0.070986
...,...,...,...,...
480,CUS_0x8234,2024-01-01,XGB_model_2024_06_01.pkl,0.397824
481,CUS_0x7dac,2024-01-01,XGB_model_2024_06_01.pkl,0.041954
482,CUS_0x7eb0,2024-01-01,XGB_model_2024_06_01.pkl,0.031617
483,CUS_0x673c,2024-01-01,XGB_model_2024_06_01.pkl,0.217473


### Save Model Inference to Datamart Gold Table

In [18]:
# create gold directory 
gold_directory = f"datamart/gold/model_predictions/{config["model_name"][:-4]}/"
print(gold_directory)

if not os.path.exists(gold_directory):
    os.makedirs(gold_directory)

# save gold table - IRL connect to database to write
partition_name = config["model_name"][:-4] + "_predictions_" + snapshot_date_str.replace('-','_') + '.parquet'
filepath = gold_directory + partition_name
spark.createDataFrame(y_inference_pdf).write.mode("overwrite").parquet(filepath)
# df.toPandas().to_parquet(filepath,
#           compression='gzip')
print('saved to:', filepath)

datamart/gold/model_predictions/XGB_model_2024_06_01/


saved to: datamart/gold/model_predictions/XGB_model_2024_06_01/XGB_model_2024_06_01_predictions_2024_01_01.parquet


### Backfill Model Predictions


In [19]:
# set up config
snapshot_date_str = "2023-01-01"

start_date_str = "2023-01-01"
end_date_str = "2024-12-01"

In [ ]:
# generate list of dates to process
def generate_first_of_month_dates(start_date_str, end_date_str):
    # Convert the date strings to datetime objects
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    
    # List to store the first of month dates
    first_of_month_dates = []

    # Start from the first of the month of the start_date
    current_date = datetime(start_date.year, start_date.month, 1)

    while current_date <= end_date:
        # Append the date in yyyy-mm-dd format
        first_of_month_dates.append(current_date.strftime("%Y-%m-%d"))
        
        # Move to the first of the next month
        if current_date.month == 12:
            current_date = datetime(current_date.year + 1, 1, 1)
        else:
            current_date = datetime(current_date.year, current_date.month + 1, 1)

    return first_of_month_dates

dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)

# Assuming monthly snapshot frequency
last_available_label_month = datetime(2024, 12, 1)
cutoff_snapshot_date = last_available_label_month - relativedelta(months=6)

dates_str_lst = [d for d in dates_str_lst if datetime.strptime(d, "%Y-%m-%d") <= cutoff_snapshot_date]



In [29]:
for snapshot_date in dates_str_lst:
    print(snapshot_date)
    model_inference.main(snapshot_date, model_name)

2023-01-01


---starting job---


{'model_artefact_filepath': 'model_bank/XGB_model_2024_06_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'XGB_model_2024_06_01.pkl',
 'snapshot_date': datetime.datetime(2023, 1, 1, 0, 0),
 'snapshot_date_str': '2023-01-01'}
Model loaded successfully! model_bank/XGB_model_2024_06_01.pkl


row_count: 8974


extracted features_sdf 530 2023-01-01 00:00:00


X_inference 530
datamart/gold/model_predictions/XGB_model_2024_06_01/


saved to: datamart/gold/model_predictions/XGB_model_2024_06_01/XGB_model_2024_06_01_predictions_2023_01_01.parquet


---completed job---


2023-02-01


---starting job---


{'model_artefact_filepath': 'model_bank/XGB_model_2024_06_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'XGB_model_2024_06_01.pkl',
 'snapshot_date': datetime.datetime(2023, 2, 1, 0, 0),
 'snapshot_date_str': '2023-02-01'}
Model loaded successfully! model_bank/XGB_model_2024_06_01.pkl


...

saved to: datamart/gold/model_predictions/XGB_model_2024_06_01/XGB_model_2024_06_01_predictions_2024_05_01.parquet


---completed job---


2024-06-01


---starting job---


{'model_artefact_filepath': 'model_bank/XGB_model_2024_06_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'XGB_model_2024_06_01.pkl',
 'snapshot_date': datetime.datetime(2024, 6, 1, 0, 0),
 'snapshot_date_str': '2024-06-01'}
Model loaded successfully! model_bank/XGB_model_2024_06_01.pkl


row_count: 8974


extracted features_sdf 498 2024-06-01 00:00:00


X_inference 498
datamart/gold/model_predictions/XGB_model_2024_06_01/


saved to: datamart/gold/model_predictions/XGB_model_2024_06_01/XGB_model_2024_06_01_predictions_2024_06_01.parquet


---completed job---




### Check Datamart

In [32]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

In [38]:
folder_path = "datamart/gold/model_predictions/XGB_model_2024_06_01/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
df = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",df.count())

df.show()

row_count: 8974
+-----------+-------------+--------------------+--------------------+
|Customer_ID|snapshot_date|          model_name|   model_predictions|
+-----------+-------------+--------------------+--------------------+
| CUS_0x144c|   2024-03-01|XGB_model_2024_06...|0.005440913606435...|
|  CUS_0x7ae|   2024-03-01|XGB_model_2024_06...| 0.12873394787311554|
| CUS_0x956f|   2024-03-01|XGB_model_2024_06...|0.011422139592468739|
| CUS_0x99c9|   2024-03-01|XGB_model_2024_06...|0.009557964280247688|
| CUS_0xb51d|   2024-03-01|XGB_model_2024_06...|0.006137938238680363|
| CUS_0xbda9|   2024-03-01|XGB_model_2024_06...|  0.5743997693061829|
| CUS_0x7b62|   2024-03-01|XGB_model_2024_06...| 0.17798259854316711|
| CUS_0x929d|   2024-03-01|XGB_model_2024_06...|  0.1018320694565773|
| CUS_0x906e|   2024-03-01|XGB_model_2024_06...|  0.8867549896240234|
| CUS_0x9661|   2024-03-01|XGB_model_2024_06...| 0.10656747221946716|
| CUS_0x3b11|   2024-03-01|XGB_model_2024_06...| 0.11147800832986832|
|  C